In [ ]:
from pathlib import Path
import shutil
import zipfile
import os

DATASET_PATH = ""
SOURCE_DIR = Path("/kaggle/working/extensions_source")
WORK_DIR = Path("/kaggle/working/project5_extensions")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH:
    raise ValueError("Điền path Kaggle Dataset vào DATASET_PATH ở đầu cell này.")
DATASET_PATH = Path(DATASET_PATH)
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy Dataset path: {DATASET_PATH}")

required_files = ("experiments.py", "attention.py", "gpt_model.py", "chargpt.py", "input.txt", "data/mnist.npz", "data/lang_id.npz")

def has_extension_files(folder):
    return all((folder / name).is_file() for name in required_files)

def zip_contains_extensions(zip_path):
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = {name.lstrip("./") for name in archive.namelist()}
            basenames = {Path(name).name for name in names}
            has_mnist = any(name.endswith("data/mnist.npz") for name in names)
            has_lang_id = any(name.endswith("data/lang_id.npz") for name in names)
            required_basenames = {"experiments.py", "attention.py", "gpt_model.py", "chargpt.py", "input.txt"}
            return required_basenames.issubset(basenames) and has_mnist and has_lang_id
    except zipfile.BadZipFile:
        return False

source = None
if DATASET_PATH.is_file():
    archives = [DATASET_PATH] if zip_contains_extensions(DATASET_PATH) else []
else:
    archives = [path for path in DATASET_PATH.rglob("*.zip") if zip_contains_extensions(path)]

for archive_path in archives:
    extract_dir = SOURCE_DIR / archive_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
    candidates = [path.parent for path in extract_dir.rglob("experiments.py") if has_extension_files(path.parent)]
    if candidates:
        source = candidates[0]
        break

if source is None and DATASET_PATH.is_dir():
    candidates = [path.parent for path in DATASET_PATH.rglob("experiments.py") if has_extension_files(path.parent)]
    if candidates:
        source = candidates[0]

if source is None:
    raise FileNotFoundError("Dataset cần có extension runner, Q7 source/corpus và hai file dữ liệu, trực tiếp hoặc bên trong ZIP.")

for name in required_files:
    destination = WORK_DIR / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source / name, destination)
os.chdir(WORK_DIR)
print("Extension project:", Path.cwd())
print("Source found at:", source)

In [ ]:
import importlib.util
import subprocess
import sys

missing_packages = [name for name in ("torch", "numpy", "sklearn") if importlib.util.find_spec(name) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy<2", "torch", "scikit-learn"])

print("PyTorch:", subprocess.check_output([sys.executable, "-c", "import torch; print(torch.__version__)"]).decode().strip())

In [ ]:
import json
import subprocess
import time
from pathlib import Path

REPORT_DIR = Path("/kaggle/working/project5_extension_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
experiments = ("q3_svm", "q4_cnn")
results = []

for experiment in experiments:
    log_path = REPORT_DIR / f"{experiment}.log"
    print(f"\n========== RUN {experiment.upper()} EXTENSION ==========")
    started = time.perf_counter()
    process = subprocess.Popen(
        [sys.executable, "-u", "experiments.py", "--question", experiment, "--project-dir", str(WORK_DIR)],
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    with log_path.open("w", encoding="utf-8") as log_file:
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
    return_code = process.wait()
    elapsed = time.perf_counter() - started
    log_text = log_path.read_text(encoding="utf-8", errors="replace")
    result_lines = [line[len("RESULT "):].strip() for line in log_text.splitlines() if line.startswith("RESULT ")]
    result = json.loads(result_lines[-1]) if result_lines else {"experiment": experiment, "error": f"exit code {return_code}"}
    result["wall_seconds"] = elapsed
    result["return_code"] = return_code
    result["log"] = str(log_path)
    results.append(result)
    print(f"{experiment.upper()} finished in {elapsed:.1f}s; log: {log_path}")

q7_log = REPORT_DIR / "q7_character_gpt.log"
print("\n========== RUN Q7 CHARACTER GPT ==========")
started = time.perf_counter()
process = subprocess.Popen(
    [sys.executable, "-u", "chargpt.py"],
    cwd=WORK_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
with q7_log.open("w", encoding="utf-8") as log_file:
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
q7_return_code = process.wait()
q7_elapsed = time.perf_counter() - started
results.append({"experiment": "Q7 Character-GPT", "train_seconds": q7_elapsed, "return_code": q7_return_code, "log": str(q7_log)})
print(f"Q7 finished in {q7_elapsed:.1f}s; log: {q7_log}")

summary = ["Project 5 — extensions only", ""]
for result in results:
    summary.append(
        f"{result.get('experiment', 'unknown')}: test accuracy {result.get('test_accuracy', 'n/a')}; "
        f"train time {result.get('train_seconds', 'n/a')}s; parameters {result.get('parameter_count', 'n/a')}; "
        f"exit code {result.get('return_code', 'n/a')}"
    )
    summary.append(f"Log: {result.get('log', 'n/a')}")
(REPORT_DIR / "summary.txt").write_text("\n".join(summary), encoding="utf-8")
print("\n" + "\n".join(summary))
print("Reports:", REPORT_DIR)